# AI-Powered Banking Fraud Detection System
## Exploratory Data Analysis & Machine Learning Pipeline

This notebook covers:
- Data Preprocessing
- Exploratory Data Analysis (EDA)
- Model Training (Isolation Forest & Random Forest)
- Evaluation & Visualizations

---
## 1. Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib
import os

# Set style for visualizations
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('ggplot')
sns.set_palette('husl')

In [ ]:
# Load dataset - Update path if your file is elsewhere
dataset_path = '../dataset/creditcard.csv'

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
else:
    # Create sample data for demonstration (use real dataset from Kaggle)
    print(f"Dataset not found at {dataset_path}")
    print("Please download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud")
    print("Place creditcard.csv in the dataset/ folder")
    df = None

---
## 2. Data Preprocessing

In [ ]:
if df is not None:
    # Basic info
    print("="*60)
    print("DATASET INFO")
    print("="*60)
    df.info()
    print("\n")
    print("\nFirst 5 rows:")
    display(df.head())

In [ ]:
if df is not None:
    # Handle missing values
    print("Missing values per column:")
    print(df.isnull().sum())
    print("\nNo missing values - dataset is clean!" if df.isnull().sum().sum() == 0 else "\nHandling missing values...")
    
    # Normalize Amount using StandardScaler
    scaler = StandardScaler()
    df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
    
    # Class distribution
    print("\n" + "="*60)
    print("CLASS DISTRIBUTION (Class imbalance)")
    print("="*60)
    print(df['Class'].value_counts())
    print(f"\nFraud %: {df['Class'].mean()*100:.4f}%")
    
    # Save scaler for later use
    joblib.dump(scaler, '../models/amount_scaler.pkl')

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Fraud vs Non-Fraud Distribution (Bar Chart)

In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    counts = df['Class'].value_counts()
    bars = ax.bar(['Legitimate (0)', 'Fraud (1)'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
    ax.set_title('Transaction Class Distribution: Fraud vs Non-Fraud', fontsize=14, fontweight='bold')
    ax.set_ylabel('Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, str(int(bar.get_height())), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visuals/01_fraud_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print("\nInsight: Severe class imbalance - fraud transactions are rare (~0.17%).")
    print("This requires careful handling (undersampling/oversampling or class weights).")

### 3.2 Transaction Amount Distribution (Histogram)

In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # All transactions
    axes[0].hist(df['Amount'], bins=50, color='#3498db', edgecolor='black', alpha=0.7)
    axes[0].set_title('Transaction Amount Distribution (All)')
    axes[0].set_xlabel('Amount')
    
    # Log scale for better view
    axes[1].hist(df['Amount'], bins=50, color='#9b59b6', edgecolor='black', alpha=0.7, log=True)
    axes[1].set_title('Transaction Amount (Log Scale)')
    axes[1].set_xlabel('Amount')
    plt.tight_layout()
    plt.savefig('../visuals/02_amount_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print("\nInsight: Most transactions are small amounts; distribution is right-skewed.")
    print("Fraudulent transactions may have different amount patterns.")

### 3.3 Correlation Heatmap

In [ ]:
if df is not None:
    # Select key features for correlation
    corr_cols = ['Time', 'Amount', 'V1', 'V2', 'V3', 'V4', 'V5', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'Class']
    corr_df = df[corr_cols].corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_df, annot=False, cmap='RdBu_r', center=0, square=True, linewidths=0.5)
    plt.title('Feature Correlation Heatmap (Key Variables)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visuals/03_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print("\nInsight: V17, V14, V12, V10 show stronger correlation with Class (fraud).")
    print("These PCA components help distinguish fraudulent patterns.")

### 3.4 Fraud Transactions Over Time

In [ ]:
if df is not None:
    # Aggregate fraud by time (binned)
    df['Time_bin'] = pd.cut(df['Time'], bins=50)
    time_fraud = df.groupby('Time_bin')['Class'].agg(['sum', 'count']).reset_index()
    time_fraud['Time_num'] = range(len(time_fraud))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(time_fraud['Time_num'], time_fraud['sum'], color='#e74c3c', linewidth=2, label='Fraud Count')
    ax.fill_between(time_fraud['Time_num'], time_fraud['sum'], alpha=0.3, color='#e74c3c')
    ax.set_title('Fraudulent Transactions Over Time', fontsize=14, fontweight='bold')
    ax.set_xlabel('Time Bins')
    ax.set_ylabel('Number of Fraud Transactions')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../visuals/04_fraud_over_time.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print("\nInsight: Fraud may cluster at certain times - useful for temporal pattern detection.")

---
## 4. Feature Preparation & Train/Test Split

In [ ]:
if df is not None:
    # Features and target
    feature_cols = [f'V{i}' for i in range(1, 29)] + ['Amount_scaled', 'Time']
    X = df[feature_cols]
    y = df['Class']
    
    # Stratified split to preserve class ratio
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print(f"Training set: {X_train.shape[0]:,} samples")
    print(f"Test set: {X_test.shape[0]:,} samples")
    print(f"Training fraud %: {y_train.mean()*100:.4f}%")
    print(f"Test fraud %: {y_test.mean()*100:.4f}%")

---
## 5. Model Training

### 5.1 Isolation Forest (Primary Anomaly Detection Model)

In [ ]:
if df is not None:
    # Isolation Forest: -1 = anomaly (fraud), 1 = normal
    iso_forest = IsolationForest(
        n_estimators=100,
        contamination=0.002,  # ~0.17% fraud rate
        random_state=42,
        max_samples='auto'
    )
    iso_forest.fit(X_train)
    
    # Convert predictions: -1 -> 1 (fraud), 1 -> 0 (legitimate)
    iso_pred = iso_forest.predict(X_test)
    iso_pred_binary = np.where(iso_pred == -1, 1, 0)
    iso_scores = -iso_forest.score_samples(X_test)  # Higher = more anomalous
    
    # Anomaly scores (normalized to 0-1 as fraud probability)
    iso_prob = (iso_scores - iso_scores.min()) / (iso_scores.max() - iso_scores.min() + 1e-8)
    
    # Save model
    joblib.dump(iso_forest, '../models/isolation_forest.pkl')
    print("Isolation Forest trained and saved.")
    
    print("\n--- Isolation Forest Metrics ---")
    print(f"Accuracy:  {accuracy_score(y_test, iso_pred_binary):.4f}")
    print(f"Precision: {precision_score(y_test, iso_pred_binary, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, iso_pred_binary, zero_division=0):.4f}")
    print(f"F1 Score:  {f1_score(y_test, iso_pred_binary, zero_division=0):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, iso_pred_binary))

### 5.2 Random Forest Classifier

In [ ]:
if df is not None:
    # Use class_weight='balanced' to handle imbalance
    rf_model = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        max_depth=10
    )
    rf_model.fit(X_train, y_train)
    
    rf_pred = rf_model.predict(X_test)
    rf_prob = rf_model.predict_proba(X_test)[:, 1]
    
    joblib.dump(rf_model, '../models/random_forest.pkl')
    print("Random Forest trained and saved.")
    
    print("\n--- Random Forest Metrics ---")
    print(f"Accuracy:  {accuracy_score(y_test, rf_pred):.4f}")
    print(f"Precision: {precision_score(y_test, rf_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, rf_pred, zero_division=0):.4f}")
    print(f"F1 Score:  {f1_score(y_test, rf_pred, zero_division=0):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, rf_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, rf_pred, target_names=['Legitimate', 'Fraud']))

### 5.3 Feature Importance (Random Forest)

In [ ]:
if df is not None:
    importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=True)
    
    plt.figure(figsize=(10, 8))
    plt.barh(importance['feature'], importance['importance'], color='#3498db')
    plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('../visuals/05_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print("\nInsight: V17, V14, V12, V10, V11 are among the most important for fraud detection.")

---
## 6. Fraud Probability & Anomaly Score Visualization

In [ ]:
if df is not None:
    # Fraud probability distribution (Random Forest)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(rf_prob[y_test==0], bins=50, alpha=0.7, label='Legitimate', color='green')
    axes[0].hist(rf_prob[y_test==1], bins=50, alpha=0.7, label='Fraud', color='red')
    axes[0].set_title('Fraud Probability Distribution (Random Forest)')
    axes[0].set_xlabel('Fraud Probability')
    axes[0].legend()
    
    axes[1].hist(iso_prob[y_test==0], bins=50, alpha=0.7, label='Legitimate', color='green')
    axes[1].hist(iso_prob[y_test==1], bins=50, alpha=0.7, label='Fraud', color='red')
    axes[1].set_title('Anomaly Score (Isolation Forest)')
    axes[1].set_xlabel('Normalized Anomaly Score')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig('../visuals/06_fraud_probability.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

---
## 7. Interactive Plotly Dashboard

In [ ]:
if df is not None:
    # Interactive correlation with Class
    class_corr = df[feature_cols + ['Class']].corr()['Class'].drop('Class').sort_values(ascending=True)
    fig = px.bar(x=class_corr.values, y=class_corr.index, orientation='h',
                 title='Feature Correlation with Fraud (Class)',
                 labels={'x': 'Correlation', 'y': 'Feature'})
    fig.update_layout(height=500, template='plotly_white', font=dict(size=12))
    fig.show()

In [ ]:
if df is not None:
    # Save feature columns for inference
    joblib.dump(feature_cols, '../models/feature_columns.pkl')
    print("Feature columns saved for inference.")
    print("\n✓ All models and visualizations complete!")